In [ ]:
# For tips on running notebooks in Google Colab, see
# https://pytorch.org/tutorials/beginner/colab
%matplotlib inline

Neural Networks
===============

Neural networks can be constructed using the `torch.nn` package.

Now that you had a glimpse of `autograd`, `nn` depends on `autograd` to
define models and differentiate them. An `nn.Module` contains layers,
and a method `forward(input)` that returns the `output`.

For example, look at this network that classifies digit images:

![convnet](https://pytorch.org/tutorials/_static/img/mnist.png)

It is a simple feed-forward network. It takes the input, feeds it
through several layers one after the other, and then finally gives the
output.

A typical training procedure for a neural network is as follows:

-   Define the neural network that has some learnable parameters (or
    weights)
-   Iterate over a dataset of inputs
-   Process input through the network
-   Compute the loss (how far is the output from being correct)
-   Propagate gradients back into the network's parameters
-   Update the weights of the network, typically using a simple update
    rule: `weight = weight - learning_rate * gradient`

Define the network
------------------

Let's define this network with number of neurons defined below:


In [ ]:
# TODO

import torch
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()

        # an affine operation: y = Wx + b
        self.fc1 =  nn.Linear(28*28, 1024)      # nn.Linear layer
        self.fc2 =        # nn.Linear layer
        self.fc3 =        # nn.Linear layer
    def forward(self, input):
        # Flatten image into (N, num_pixels) dimensions
        out = self.fc1(x) # N, 128
        out = F.relu(out)

        # Apply first layer. Apply Relu to output


        # Fully connected layer Fc2: (N, 120) Tensor input,
        # and outputs a (N, 84) Tensor, it uses RELU activation function

        # Final layer OUTPUT: (N, 84) Tensor input, and
        # outputs a (N, 10) Tensor


        return output


net = Net()
print(net)

You just have to define the `forward` function, and the `backward`
function (where gradients are computed) is automatically defined for you
using `autograd`. You can use any of the Tensor operations in the
`forward` function.

The learnable parameters of a model are returned by `net.parameters()`


In [ ]:
params = list(net.parameters())
print(len(params))
print(params[1].size())  # conv1's .weight

Let\'s try a random 32x32 input. Note: expected input size of this net
(LeNet) is 32x32. To use this net on the MNIST dataset, please resize
the images from the dataset to 32x32.


In [ ]:
input = torch.randn(1, 1, 32, 32)
out = net(input)
print(out)

Zero the gradient buffers of all parameters and backprops with random
gradients:


In [ ]:
net.zero_grad()
out.backward(torch.randn(1, 10))

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>
<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">
<p><code>torch.nn</code> only supports mini-batches. The entire <code>torch.nn</code>package only supports inputs that are a mini-batch of samples, and nota single sample.For example, <code>nn.Conv2d</code> will take in a 4D Tensor of<code>nSamples x nChannels x Height x Width</code>.If you have a single sample, just use <code>input.unsqueeze(0)</code> to adda fake batch dimension.</p>
</div>

Before proceeding further, let\'s recap all the classes you've seen so
far.

**Recap:**

:   -   `torch.Tensor` - A *multi-dimensional array* with support for
        autograd operations like `backward()`. Also *holds the gradient*
        w.r.t. the tensor.
    -   `nn.Module` - Neural network module. *Convenient way of
        encapsulating parameters*, with helpers for moving them to GPU,
        exporting, loading, etc.
    -   `nn.Parameter` - A kind of Tensor, that is *automatically
        registered as a parameter when assigned as an attribute to a*
        `Module`.
    -   `autograd.Function` - Implements *forward and backward
        definitions of an autograd operation*. Every `Tensor` operation
        creates at least a single `Function` node that connects to
        functions that created a `Tensor` and *encodes its history*.

**At this point, we covered:**

:   -   Defining a neural network
    -   Processing inputs and calling backward

**Still Left:**

:   -   Computing the loss
    -   Updating the weights of the network

Loss Function
=============

A loss function takes the (output, target) pair of inputs, and computes
a value that estimates how far away the output is from the target.

There are several different [loss
functions](https://pytorch.org/docs/nn.html#loss-functions) under the nn
package . A simple loss is: `nn.MSELoss` which computes the mean-squared
error between the output and the target.

For example:


In [ ]:
output = net(input)
target = torch.randn(10)  # a dummy target, for example
target = target.view(1, -1)  # make it the same shape as output
criterion = nn.MSELoss()

loss = criterion(output, target)
print(loss)

Now, if you follow `loss` in the backward direction, using its
`.grad_fn` attribute, you will see a graph of computations that looks
like this:

``` {.sourceCode .sh}
input -> conv2d -> relu -> maxpool2d -> conv2d -> relu -> maxpool2d
      -> flatten -> linear -> relu -> linear -> relu -> linear
      -> MSELoss
      -> loss
```

So, when we call `loss.backward()`, the whole graph is differentiated
w.r.t. the neural net parameters, and all Tensors in the graph that have
`requires_grad=True` will have their `.grad` Tensor accumulated with the
gradient.

For illustration, let us follow a few steps backward:


In [ ]:
print(loss.grad_fn)  # MSELoss
print(loss.grad_fn.next_functions[0][0])  # Linear
print(loss.grad_fn.next_functions[0][0].next_functions[0][0])  # ReLU

In [ ]:
# You may need to install torchviz:
!pip install torchviz
import contextlib

try:
    from torchviz import make_dot
    # Visualizing the computation graph for `out`
    display(make_dot(loss, params={'input': input}))
except ImportError:
    print("Please install torchviz using `!pip install torchviz` to view the computational graph.")

Backprop
========

To backpropagate the error all we have to do is to `loss.backward()`.
You need to clear the existing gradients though, else gradients will be
accumulated to existing gradients.

Now we shall call `loss.backward()`, and have a look at conv1\'s bias
gradients before and after the backward.


In [ ]:
net.zero_grad()     # zeroes the gradient buffers of all parameters

print('fc1.bias.grad before backward')
print(net.fc1.bias.grad)

loss.backward()

print('fc2.bias.grad after backward')
print(net.fc1.bias.grad)

Now, we have seen how to use loss functions.

**Read Later:**

> The neural network package contains various modules and loss functions
> that form the building blocks of deep neural networks. A full list
> with documentation is [here](https://pytorch.org/docs/nn).

**The only thing left to learn is:**

> -   Updating the weights of the network

Update the weights
==================

The simplest update rule used in practice is the Stochastic Gradient
Descent (SGD):

``` {.sourceCode .python}
weight = weight - learning_rate * gradient
```

We can implement this using simple Python code:

``` {.sourceCode .python}
learning_rate = 0.01
for f in net.parameters():
    f.data.sub_(f.grad.data * learning_rate)
```

However, as you use neural networks, you want to use various different
update rules such as SGD, Nesterov-SGD, Adam, RMSProp, etc. To enable
this, we built a small package: `torch.optim` that implements all these
methods. Using it is very simple:

``` {.sourceCode .python}
import torch.optim as optim

# create your optimizer
optimizer = optim.SGD(net.parameters(), lr=0.01)

# in your training loop:
optimizer.zero_grad()   # zero the gradient buffers
output = net(input)
loss = criterion(output, target)
loss.backward()
optimizer.step()    # Does the update
```


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>
<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">
<p>Observe how gradient buffers had to be manually set to zero using<code>optimizer.zero_grad()</code>. This is because gradients are accumulatedas explained in the <a href="">Backprop</a> section.</p>
</div>


In [ ]:
import torchvision
import torchvision.transforms as transforms

Custom Datasets & DataLoaders
-----------------------------
Instead of relying completely on the black-box `torchvision.datasets.MNIST` loader, let's create a **custom Dataset class** manually. This will help you understand exactly what PyTorch expects under the hood:
1. `__init__`: Runs once to initialize data.
2. `__len__`: Returns the total size of your dataset.
3. `__getitem__`: Returns a single sample (image and label) given an index.

We will also configure the download directory (`BASE_DIR`) to adjust dynamically depending on whether you are running this in Google Colab or on your local machine.

In [ ]:
import os
import sys
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from PIL import Image

# 1. Configure the download path based on the environment (Colab vs Offline)
if 'google.colab' in sys.modules:
    BASE_DIR = '/content/data'
else:
    BASE_DIR = './data'

os.makedirs(BASE_DIR, exist_ok=True)

# Fetch the raw dataset items directly to use for our custom Dataset
raw_train = datasets.MNIST(root=BASE_DIR, train=True, download=True)
raw_test = datasets.MNIST(root=BASE_DIR, train=False, download=True)


In [ ]:
# 2. Define our custom Dataset
class CustomMNIST(Dataset):
    def __init__(self, raw_data, transform=None):
        # We explicitly extract the images and labels
        self.images = raw_data.data
        self.labels = raw_data.targets
        self.transform = transform

    def __len__(self):
        # PyTorch needs to know the total number of samples
        return len(self.labels)

    def __getitem__(self, idx):
        # Fetch the specific index
        img = self.images[idx].numpy()
        label = self.labels[idx]

        # Convert back to PIL Image so our torchvision transforms work normally
        img = Image.fromarray(img, mode='L')

        if self.transform:
            img = self.transform(img)

        return img, label

# 3. Setting dataset transforms
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize 28x28 to 32x32 for LeNet structure
    transforms.ToTensor(),        # Convert PIL to PyTorch Tensor in range [0, 1]
    # transforms.Normalize((0.5,), (0.5,)) # Optional standardization
])

batch_size = 4

# 4. Instantiate our manual datasets
trainset = CustomMNIST(raw_train, transform=transform)
testset = CustomMNIST(raw_test, transform=transform)

# 5. Create DataLoaders to handle batching and shuffling
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
# Define loss and optimizer
import torch.optim as optim

net = Net()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.01, weight_decay=0.01)

## SUGGESTION: Play with value of lr (increase/decrease it by 10x ) to get best possible accuracy

In [ ]:
# Select device
device = 'cpu'
net.to(device)

Data Visualization
------------------
Before training, it's always a good idea to visualize the dataset to understand what the model will be learning. Let's visualize a few samples of the MNIST dataset along with their labels.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# A helper function to un-normalize and display an image
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)), cmap='gray')

# Get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Define classes for MNIST
classes = tuple(str(i) for i in range(10))

# Show images in a grid
fig = plt.figure(figsize=(10, 4))
for idx in np.arange(batch_size):
    ax = fig.add_subplot(1, batch_size, idx+1, xticks=[], yticks=[])
    imshow(images[idx])
    ax.set_title(classes[labels[idx]])
plt.show()

In [ ]:
import time

start_time = time.time()

for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

end_time = time.time()
mlp_train_time = end_time - start_time
print('Finished Training')
print(f'Time taken to train MLP: {mlp_train_time:.2f} seconds')

In [ ]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

mlp_accuracy = 100 * correct / total
print(f'Accuracy of the MLP network on the 10000 test images: {mlp_accuracy:.2f} %')

In [ ]:
import matplotlib.pyplot as plt

_, ax = plt.subplots(1, 5, figsize=(40,20))
for i in range(5):
  ax[i].imshow(trainset[i][0][0], cmap='gray')

## Visualize weights

In [ ]:
#TODO

import matplotlib.pyplot as plt

# Extract fc1 weights of shape 120 * 1024. You may want to use net.parameters()
fc1_weights =

# Reshape them to 32 x 32 image. Bring to CPU, convert to Numpy if needed
fc1_weights =

# There will be 120 of such images for fc1 weight. We visualize some of them.
_, ax = plt.subplots(1, 5, figsize=(20,20))
for i in range(5):
  ax[i].imshow(fc1_weights[i],)

## Analyzing Failure Cases
Instead of just looking at the accuracy number, it's very helpful to visualize exactly where our model is making mistakes. Let's plot some of the images where our MLP network predicted the wrong digit.

In [ ]:
import matplotlib.pyplot as plt

net.eval()
misclassified_images = []
misclassified_preds = []
true_labels = []

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)

        # Find where predictions are incorrect
        incorrect_idx = (predicted != labels).nonzero(as_tuple=True)[0]

        for idx in incorrect_idx:
            misclassified_images.append(images[idx].cpu().squeeze().numpy())
            misclassified_preds.append(predicted[idx].item())
            true_labels.append(labels[idx].item())

            # We just want to visualize a few of them
            if len(misclassified_images) == 5:
                break
        if len(misclassified_images) == 5:
            break

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    axes[i].imshow(misclassified_images[i], cmap='gray')
    axes[i].set_title(f"Pred: {misclassified_preds[i]}\nTrue: {true_labels[i]}", color="red")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

Convolutional Neural Networks (LeNet-5)
=====================================

Now we will implement **LeNet-5**, one of the earliest Convolutional Neural Networks (CNNs) designed by Yann LeCun for handwritten digit recognition (like MNIST).

Unlike our previous MLP that completely flattened the 2D image into a 1D vector (destroying the spatial relationship of pixels), a CNN preserves the 2D structure by using Convolutional layers (`nn.Conv2d`). It sweeps small matrices (kernels/filters) across the image to extract features like edges, curves, and angles.

### Architecture of LeNet-5:
1. **Conv1**: Applies 6 filters of size 5x5 to the 1-channel image.
2. **Subsampling (Pool1)**: Reduces spatial dims by half using a 2x2 Max Pooling window.
3. **Conv2**: Applies 16 filters of size 5x5.
4. **Subsampling (Pool2)**: Max Pooling (2x2).
5. **Flatten**: Converts the 3D feature maps into a 1D vector.
6. **FC1 (Linear)**: Fully connected layer to 120 nodes.
7. **FC2 (Linear)**: Fully connected layer to 84 nodes.
8. **Output (Linear)**: 10 nodes for our 10 digit classes.

In [ ]:
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        # 1 input image channel, 6 output channels, 5x5 square convolution
        self.conv1 = nn.Conv2d(1, 6, 5)
        # 6 input channels, 16 output channels, 5x5 square convolution
        self.conv2 = nn.Conv2d(6, 16, 5)

        # an affine operation: y = Wx + b
        # In our case, after 2 max pooling operations (each dividing dimensions by 2),
        # an initial 32x32 image becomes 5x5 structurally.
        # Wait - MNIST is originally 28x28, but we resized it earlier using transforms.Resize((32,32))!
        # So 32 -> conv(5x5) -> 28 -> maxpool(2x2) -> 14 -> conv(5x5) -> 10 -> maxpool(2x2) -> 5
        # Total flattened nodes: 16 channels * 5 * 5 = 400
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # If the size is a square you can only specify a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        # Flatten all dimensions except the batch dimension
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

lenet = LeNet()
lenet.to(device)
print(lenet)

Visualizing the Computation Graph for LeNet
-----------------------------------------
Let's trace a dummy input through our defined CNN and witness the backpropagation paths generated logically by PyTorch!

In [ ]:
dummy_cnn_input = torch.randn(1, 1, 32, 32).to(device)
cnn_out = lenet(dummy_cnn_input)
cnn_loss = criterion(cnn_out, torch.empty(1, dtype=torch.long).random_(10).to(device))

try:
    from torchviz import make_dot
    display(make_dot(cnn_loss, params=dict(lenet.named_parameters())))
except ImportError:
    print("Please install torchviz using `!pip install torchviz` to view the computational graph.")

Training LeNet-5
----------------

Here, we will define a new optimizer for our CNN, train it for 2 epochs, and explicitly track the time we spent training!

In [ ]:
cnn_optimizer = optim.Adam(lenet.parameters(), lr=0.001)

start_time = time.time()

for epoch in range(2):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        cnn_optimizer.zero_grad()

        outputs = lenet(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        cnn_optimizer.step()

        running_loss += loss.item()
        if i % 2000 == 1999:
            print(f'[LeNet] [{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

end_time = time.time()
cnn_train_time = end_time - start_time
print('Finished Training LeNet-5')
print(f'Time taken to train LeNet: {cnn_train_time:.2f} seconds')

In [ ]:
cnn_correct = 0
cnn_total = 0

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = lenet(images)
        _, predicted = torch.max(outputs.data, dim=1)
        cnn_total += labels.size(0)
        cnn_correct += (predicted == labels).sum().item()

cnn_accuracy = 100 * cnn_correct / cnn_total
print(f'Accuracy of LeNet-5 on the 10000 test images: {cnn_accuracy:.2f} %')

## Analyzing Failure Cases for LeNet-5
Just like we did for the MLP, let's check which images our Convolutional Neural Network struggled to predict correctly. Because CNNs are much more accurate on images, the mistakes it *does* make are usually much harder to recognize even for a human!

In [ ]:
import matplotlib.pyplot as plt

lenet.eval() # Set to evaluation mode
cnn_misclassified_images = []
cnn_misclassified_preds = []
cnn_true_labels = []

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = lenet(images)
        _, predicted = torch.max(outputs, 1)

        # Find where predictions are incorrect
        incorrect_idx = (predicted != labels).nonzero(as_tuple=True)[0]

        for idx in incorrect_idx:
            cnn_misclassified_images.append(images[idx].cpu().squeeze().numpy())
            cnn_misclassified_preds.append(predicted[idx].item())
            cnn_true_labels.append(labels[idx].item())

            if len(cnn_misclassified_images) == 5:
                break
        if len(cnn_misclassified_images) == 5:
            break

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    # Ensure there's actually a misclassified image to show (if accuracy is 100%, list could be empty)
    if i < len(cnn_misclassified_images):
        axes[i].imshow(cnn_misclassified_images[i], cmap='gray')
        axes[i].set_title(f"Pred: {cnn_misclassified_preds[i]}\nTrue: {cnn_true_labels[i]}", color="red")
        axes[i].axis('off')

plt.tight_layout()
plt.show()

MLP vs CNN Comparison
=====================
Let us computationally analyze which model is better suited for analyzing image data, based on their parameter footprint, time to converge, and final test accuracy!

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

mlp_params = count_parameters(net)
cnn_params = count_parameters(lenet)

print("="*40)
print(f"| Model | Params   | Time (s) | Acc (%) |")
print("="*40)
try:
    print(f"| MLP   | {mlp_params:8d} | {mlp_train_time:8.2f} | {mlp_accuracy:7.2f} |")
except NameError:
    print(f"| MLP   | {mlp_params:8d} | {'N/A':>8} | {'N/A':>7} |")

try:
    print(f"| LeNet | {cnn_params:8d} | {cnn_train_time:8.2f} | {cnn_accuracy:7.2f} |")
except NameError:
    print(f"| LeNet | {cnn_params:8d} | {'N/A':>8} | {'N/A':>7} |")
print("="*40)

print("\nTakeaways:")
print(f"1. LeNet has {(mlp_params/cnn_params):.1f}x FEWER parameters than the MLP!")
print("   (Because CNN filters are reused across the entire image, whereas MLP needs a connected weight for every single pixel!)")
print("2. CNNs generally take slightly longer per epoch due to the sliding mathematical operations.")
print("3. By keeping 2D spatial relationships, CNNs crush flat MLPs in accuracy on visual datasets.")